In [30]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import json
import re

In [31]:
file_path = 'election-stations-2569.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [32]:
stations_list = []
for province in data.get('provinces', []):
    if province['name'] == "ลำปาง":
        for area in province.get('areas', []):
            if area['area'] == 2:
                stations_list = area.get('stations', [])
                break

In [33]:
stations_list[:5]

[{'code': '52-02-520109-001',
  'name': 'ศาลาอเนกประสงค์ประจำหมู่บ้าน',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'},
 {'code': '52-02-520109-002',
  'name': 'ศาลาอเนกประสงค์ประจำหมู่บ้านบ้านสบมาย',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'},
 {'code': '52-02-520109-003',
  'name': 'ศาลาอเนกประสงค์ประจำหมู่บ้าน',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'},
 {'code': '52-02-520109-004',
  'name': 'ศาลาวัดแม่อาง',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'},
 {'code': '52-02-520109-005',
  'name': 'ศาลาอเนกประสงค์โรงเรียนบ้านดง (เดิม)',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'}]

In [34]:
len(stations_list)

342

In [ ]:
csv_file = 'unit_village.csv'
df = pd.read_csv(csv_file)

In [37]:
df.head(5)

,Column1,province,district,sub-district,unit_number,village,place,latitude,longitude
0,1,ลำปาง,งาว,นาแก,1,1,NaN,NaN,NaN
1,2,ลำปาง,งาว,นาแก,2,2,NaN,NaN,NaN
2,3,ลำปาง,งาว,นาแก,3,3,NaN,NaN,NaN
3,4,ลำปาง,งาว,นาแก,4,4,NaN,NaN,NaN
4,5,ลำปาง,งาว,นาแก,5,5,NaN,NaN,NaN


In [ ]:
geolocator = Nominatim(user_agent="lampang_v4_complete")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

for index, row in df.iterrows():
    c_dist = str(row['district']).strip()
    c_subdist = str(row['sub-district']).strip()
    c_unit = str(row['unit_number']).strip().zfill(3)

    # ค้นหาชื่อสถานที่จาก stations_list
    target_name = None
    for s in stations_list:
        json_unit = s['code'].split('-')[-1].strip().zfill(3)
        json_dist = s['district'].strip()
        json_subdist = s['subdistrict'].strip()

        # เงื่อนไขการ Match: เช็คทั้ง อำเภอ, ตำบล และหน่วย
        if json_unit == c_unit and json_subdist == c_subdist and json_dist == c_dist:
            target_name = s['name']
            break

    if target_name:
        df.at[index, 'place'] = target_name

    #     # ค้นหาพิกัด (Geocoding)
    #     v_num = str(row['village']).strip() if pd.notna(row['village']) else ""
    #     v_text = f"หมู่ที่ {v_num}" if v_num else ""
    #     query = f"{target_name} {v_text} {c_subdist} {c_dist} ลำปาง"

    #     try:
    #         print(f"Matching: {c_subdist} หน่วย {c_unit} -> {target_name}")
    #         location = geolocator.geocode(query)
    #         if location:
    #             df.at[index, 'latitude'] = location.latitude
    #             df.at[index, 'longitude'] = location.longitude
    #     except:
    #         pass

    #     time.sleep(1)
    # else:
    #     print(f"ไม่พบข้อมูลสำหรับ: {c_subdist} หน่วยที่ {c_unit}")

/tmp/ipykernel_5512/4234976404.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ศาลาเอนกประสงค์  บ้านทุ่งศาลา' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, 'place'] = target_name


Matching: นาแก หน่วย 001 -> ศาลาเอนกประสงค์  บ้านทุ่งศาลา
Matching: นาแก หน่วย 002 -> ศาลาประชาคม วัดหนองเหียง
Matching: นาแก หน่วย 003 -> ศาลาวัดนาแก
Matching: นาแก หน่วย 004 -> ศาลากลางหมู่บ้าน บ้านแม่ฮ่าง
Matching: นาแก หน่วย 005 -> โรงเรียนแม่แป้นวิทยา
Matching: นาแก หน่วย 006 -> ศาลากลางหมู่บ้าน บ้านสันติสุข
Matching: บ้านร้อง หน่วย 001 -> ศาลากลางหมู่บ้าน บ้านนาแรม
Matching: บ้านร้อง หน่วย 002 -> ศาลากลางหมู่บ้าน บ้านร้อง
Matching: บ้านร้อง หน่วย 003 -> ศาลาวัดสบป๋อน
Matching: บ้านร้อง หน่วย 004 -> ศาลาประชาคมบ้านข่อย
Matching: บ้านร้อง หน่วย 005 -> ศาลาวัดบ้านผาแดง
Matching: บ้านร้อง หน่วย 006 -> ศาลากลางหมู่บ้าน บ้านปากบอก
Matching: บ้านร้อง หน่วย 007 -> ศาลาประชาคมหมู่บ้าน
Matching: บ้านร้อง หน่วย 008 -> โรงเรียนบ้านใหม่พัฒนา
Matching: บ้านร้อง หน่วย 009 -> ศาลากลางหมู่บ้าน บ้านท่าเจริญ


Matching: บ้านร้อง หน่วย 010 -> ศาลากลางหมู่บ้าน บ้านร้องพัฒนา
Matching: บ้านร้อง หน่วย 011 -> โรงเรียนบ้านข่อยมิตรภาพที่ 110 สาขาแม่งาว
Matching: บ้านร้อง หน่วย 012 -> ศาลากลางหมู่บ้าน บ้านชนแดน
Matching: บ้านร้อง หน่วย 013 -> โรงเรียนบ้านแม่งาวใต้ สาขาแม่คำหล้า
Matching: บ้านหวด หน่วย 001 -> ศาลากลางหมู่บ้านบ้านร่องต้า
Matching: บ้านหวด หน่วย 002 -> ศูนย์พัฒนาตำบลบ้านหวด
Matching: บ้านหวด หน่วย 003 -> โรงเรียนไทยรัฐวิทยา 85
Matching: บ้านหวด หน่วย 004 -> ศาลาวัดห้วยทาก
Matching: บ้านหวด หน่วย 005 -> ศาลาวัดปางหละ
Matching: บ้านหวด หน่วย 006 -> สำนักสงฆ์แม่พร้าว
Matching: บ้านหวด หน่วย 007 -> ศาลากลางหมู่บ้าน บ้านใหม่ธานี
Matching: บ้านอ้อน หน่วย 001 -> ศาลากลางหมู่บ้าน บ้านแม่กวัก
Matching: บ้านอ้อน หน่วย 002 -> ศาลากลางหมู่บ้าน บ้านอ้อนใต้
Matching: บ้านอ้อน หน่วย 003 -> ศาลากลางหมู่บ้าน บ้านอ้อนเหนือ
Matching: บ้านอ้อน หน่วย 004 -> โรงเรียนปงมะโอ
Matching: บ้านอ้อน หน่วย 005 -> ศาลาประชาคมบ้านห้วยหก
Matching: บ้านอ้อน หน่วย 006 -> โรงเรียนบ้านอ้อน สาขาหัวทุ่ง
Matching: บ้านอ้อน หน่

Matching: บ้านแหง หน่วย 009 -> ศาลาสำนักสงฆ์ใหม่สามัคคี
Matching: บ้านโป่ง หน่วย 001 -> ศาลากลางหมู่บ้าน บ้านสบเอิม
Matching: บ้านโป่ง หน่วย 002 -> ศาลากลางหมู่บ้าน SML
Matching: บ้านโป่ง หน่วย 003 -> ศาลากลางหมู่บ้าน บ้านสบพลึง
Matching: บ้านโป่ง หน่วย 004 -> ศาลาโป่งหนองบัว
Matching: บ้านโป่ง หน่วย 005 -> ศาลากลางหมู่บ้าน บ้านเป๊าะ
Matching: บ้านโป่ง หน่วย 006 -> ศาลากลางหมู่บ้าน บ้านเหล่า
Matching: บ้านโป่ง หน่วย 007 -> ศาลากลางหมู่บ้าน บ้านหาดเชี่ยว
Matching: บ้านโป่ง หน่วย 008 -> ศาลากลางหมู่บ้าน บ้านต้นมื่น
Matching: บ้านโป่ง หน่วย 009 -> ศาลาอเนกประสงค์ หมู่ที่ 9
Matching: บ้านโป่ง หน่วย 010 -> ศาลาวัดบ้านเป๊าะ
Matching: บ้านโป่ง หน่วย 011 -> ศาลากลางหมู่บ้าน บ้านสันโค้งพัฒนา
Matching: บ้านโป่ง หน่วย 012 -> ศาลากลางหมู่บ้าน บ้านโป่งแก้ว
Matching: ปงเตา หน่วย 001 -> ศาลากลางหมู่บ้าน บ้านปงเตา
Matching: ปงเตา หน่วย 002 -> หอประชุมประจำหมู่บ้าน บ้านห้วยอูน
Matching: ปงเตา หน่วย 003 -> สำนักสงฆ์พร้าวใหม่
Matching: ปงเตา หน่วย 004 -> ศาลากลางหมู่บ้าน บ้านปันเหนือ
Matching: ปงเตา หน่ว

Matching: หลวงใต้ หน่วย 007 -> ศาลากลางหมู่บ้าน บ้านดง
Matching: หลวงใต้ หน่วย 008 -> ศาลากลางหมู่บ้าน บ้านใหม่เจริญสุข
Matching: แม่ตีบ หน่วย 001 -> โรงเรียนบ้านทุ่ง
Matching: แม่ตีบ หน่วย 002 -> ศาลาวัดดอกคำใต้
Matching: แม่ตีบ หน่วย 003 -> ศาลากลางหมู่บ้าน บ้านแม่ตีบหลวง
Matching: แม่ตีบ หน่วย 004 -> ศาลาวัดแม่ตีบหลวง
Matching: แม่ตีบ หน่วย 005 -> โรงเรียนบ้านแม่ตีบ
Matching: แม่ตีบ หน่วย 006 -> โรงเรียนบ้านงิ้วงาม
Matching: แม่ตีบ หน่วย 007 -> ศาลาอเนกประสงค์บ้านแม่งาว
Matching: ทุ่งฮั้ว หน่วย 001 -> อาคารศาลาประชาคมหมู่บ้านบ้านทุ่งปี้ หมู่ที่ 1
Matching: ทุ่งฮั้ว หน่วย 002 -> อาคารอเนกประสงค์วัดแม่ทรายเงิน หมู่ที่ 2
Matching: ทุ่งฮั้ว หน่วย 003 -> อาคารอเนกประสงค์บ้านผาดิน หมู่ที่ 3
Matching: ทุ่งฮั้ว หน่วย 004 -> อาคารอเนกประสงค์บ้านทุ่งฮั้ว หมู่ที่ 4
Matching: ทุ่งฮั้ว หน่วย 005 -> โรงเรียนบ้านวังมน หมู่ที่ 5
Matching: ทุ่งฮั้ว หน่วย 006 -> อาคารอเนกประสงค์บ้านผาแดง หมู่ที่ 6
Matching: ทุ่งฮั้ว หน่วย 007 -> ศาลาการเปรียญวัดห้วยกันทา หมู่ที่ 7
Matching: ทุ่งฮั้ว หน่วย 008 -> อาคา

...

Matching: แจ้ซ้อน หน่วย 003 -> ศาลาประชาคมบ้านดินดำ
Matching: แจ้ซ้อน หน่วย 004 -> ศาลาวัดบ้านทุ่ง
Matching: แจ้ซ้อน หน่วย 005 -> หอประชุมหมู่บ้าน
Matching: แจ้ซ้อน หน่วย 006 -> ศาลาวัดบ้านสบลี
Matching: แจ้ซ้อน หน่วย 007 -> ศาลาอเนกประสงค์หมู่บ้าน
Matching: แจ้ซ้อน หน่วย 008 -> โรงเรียนบ้านใหม่พัฒนา
Matching: แจ้ซ้อน หน่วย 009 -> ศาลาประชาคมบ้านข่วงกอม
Matching: แจ้ซ้อน หน่วย 010 -> ศาลาวัดปางต้นหนุน
Matching: แจ้ซ้อน หน่วย 011 -> ศาลาประชาคมบ้านแจ้ซ้อนเหนือ
Matching: แจ้ซ้อน หน่วย 012 -> หอประชุมโรงเรียนแจ้ซ้อนวิทยา
Matching: แจ้ซ้อน หน่วย 013 -> ศาลาหน้าวัดศรีหลวงแจ้ซ้อน
Matching: แจ้ซ้อน หน่วย 014 -> อาคารสหกรณ์การเกษตรบ้านปางม่วง
Matching: แจ้ซ้อน หน่วย 015 -> โบสถ์แม่แวน


Matching: บ้านเสด็จ หน่วย 001 -> ศาลาอเนกประสงค์บ้านทรายมูล 1
Matching: บ้านเสด็จ หน่วย 002 -> โรงเรียนบ้านจำค่า
Matching: บ้านเสด็จ หน่วย 003 -> อาคารอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านเสด็จ หน่วย 004 -> รพ.สต.บ้านทรายทอง
Matching: บ้านเสด็จ หน่วย 005 -> วัดปงอ้อม
Matching: บ้านเสด็จ หน่วย 006 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านเสด็จ หน่วย 007 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านเสด็จ หน่วย 008 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านเสด็จ หน่วย 009 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านเสด็จ หน่วย 010 -> ศาลาอเนกประสงค์ SML
Matching: บ้านเสด็จ หน่วย 011 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านเสด็จ หน่วย 012 -> ศาลาอเนกประสงค์วัดทรายเหนือ
Matching: บ้านเสด็จ หน่วย 013 -> โรงเรียนวัดเสด็จ


Matching: บ้านเสด็จ หน่วย 014 -> ศาลาประชาคม
Matching: บ้านเสด็จ หน่วย 015 -> วัดทรายมูล
Matching: บ้านเสด็จ หน่วย 016 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านเสด็จ หน่วย 017 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านเสด็จ หน่วย 018 -> สำนักสงฆ์ห้วยเดื่อ
Matching: บ้านเสด็จ หน่วย 019 -> วัดพระธาตุเสด็จ (ศาลาด้านใต้)
Matching: บ้านแลง หน่วย 001 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านแลง หน่วย 002 -> ศาลาอเนกประสงค์ประจำหมู่บ้านบ้านสบมาย
Matching: บ้านแลง หน่วย 003 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านแลง หน่วย 004 -> ศาลาวัดแม่อาง


Matching: บ้านแลง หน่วย 005 -> ศาลาอเนกประสงค์โรงเรียนบ้านดง (เดิม)
Matching: บ้านแลง หน่วย 006 -> ศาลาอเนกประสงค์วัดแตะพุทธราม
Matching: บ้านแลง หน่วย 007 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านแลง หน่วย 008 -> ศาลาอเนกประสงค์บ้านแม่อางน้ำล้อม
Matching: บ้านแลง หน่วย 009 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านแลง หน่วย 010 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านแลง หน่วย 011 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านแลง หน่วย 012 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: ทุ่งผึ้ง หน่วย 001 -> หอประชุมหน้าวัดทุ่งฮ้าง
Matching: ทุ่งผึ้ง หน่วย 002 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: ทุ่งผึ้ง หน่วย 003 -> ศาลาตลาดสด SML
Matching: ทุ่งผึ้ง หน่วย 004 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: ทุ่งผึ้ง หน่วย 005 -> ห้องประชุมโรงเรียนบ้านช่อฟ้า
Matching: ทุ่งผึ้ง หน่วย 006 -> ศาลาการเปรียญวัดแจ้คอน
Matching: ทุ่งผึ้ง หน่วย 007 -> ศาลาอเนกประสงค์บ้านใหม่สามัคคี
Matching: ทุ่งผึ้ง หน่วย 008 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: บ้านสา หน่วย 001 -> ศาลาอเนกประสง

Matching: เมืองมาย หน่วย 004 -> ศูนย์การเรียนรู้ชุมชนชาวไทยภูเขาแม่ฟ้าหลวง (บ้านแม่ก๋า)
Matching: เมืองมาย หน่วย 005 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: เมืองมาย หน่วย 006 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
Matching: เมืองมาย หน่วย 007 -> ศาลาอเนกประสงค์ประจำหมู่บ้าน
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(นอกเขต) หน่วยที่ 001
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(นอกเขต) หน่วยที่ 002
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(นอกเขต) หน่วยที่ 003
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(นอกเขต) หน่วยที่ 004
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(นอกเขต) หน่วยที่ 005
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(นอกเขต) หน่วยที่ 006
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(นอกเขต) หน่วยที่ 007
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(นอกเขต) หน่วยที่ 008
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(ในเขต) หน่วยที่ 001
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(ในเขต) หน่วยที่ 002
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(ในเขต) หน่วยที่ 003
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(ในเขต) หน่วยที่ 004
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(ในเขต) หน่วยที่ 005
ไม่พบข้อมูลสำหรับ: แจ้ห่ม(ในเขต) หน่วยที่ 006
Matching: แม่สุก หน่วย 001 -> ศาลา SML บ้านแม่สุก
Matching: แม่สุก หน่วย 002 

In [39]:
df.to_csv('unit_position_updated.csv', index=False, encoding='utf-8-sig')

In [40]:
stations_list[:5]

[{'code': '52-02-520109-001',
  'name': 'ศาลาอเนกประสงค์ประจำหมู่บ้าน',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'},
 {'code': '52-02-520109-002',
  'name': 'ศาลาอเนกประสงค์ประจำหมู่บ้านบ้านสบมาย',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'},
 {'code': '52-02-520109-003',
  'name': 'ศาลาอเนกประสงค์ประจำหมู่บ้าน',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'},
 {'code': '52-02-520109-004',
  'name': 'ศาลาวัดแม่อาง',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'},
 {'code': '52-02-520109-005',
  'name': 'ศาลาอเนกประสงค์โรงเรียนบ้านดง (เดิม)',
  'district': 'เมืองลำปาง',
  'subdistrict': 'บ้านแลง'}]

In [48]:
data = {}
count = 0
for station in stations_list:
    if station['district'] == "วังเหนือ":
        sub = station['subdistrict']
        data[sub] = data.get(sub, 0) + 1
        count +=1

data

{'ทุ่งฮั้ว': 12,
 'วังเหนือ': 12,
 'วังใต้': 7,
 'ร่องเคาะ': 18,
 'วังทอง': 9,
 'วังซ้าย': 10,
 'วังแก้ว': 7,
 'วังทรายคำ': 9}

In [47]:
count

84

# Update

- จากผลลัพธ์ที่ได้ ต.บ้านใหม่หายไป พบว่าใน json ตำบลบ้านใหม่คือตำบลวังเหนือ โดยที่ตำบลวังเหนือจริง ๆ มี 8 หน่วย ตามเอกสาร สส 5-18 จึงทำการ manual ตำบลบ้านใหม่เพิ่มลงไป 4 หน่วย
- manual ตำบลเเจ้ห่มทั้งหมดเนื่องจากในเอกสารมีเเยกในเขต นอกเขต

# File

- unit_village.csv คือ ไฟล์หน่วยทั้งหมดในลำปางเขต 2 โดยเเปลงมาจาก master_summary.csv เเล้ว manual เพิ่มหมู่ที่ของเเต่ละหน่วยโดยนำมาจากเอกสาร สส.5-18